# Notebook 08 — Recommendations

Recommendation 1 (Section 6.1) and Recommendation 2 (Section 6.2)'s cited numbers.

Recommendation 1 reuses notebook `07`'s Table 4 finding (Linear Regression, tied with
Ridge, has the best top-k ranking overlap among all four models — cited here, not
recomputed) and adds the one new piece of evidence: a single-day case study on the
busiest test-period day. Recommendation 2 computes, for all four models, how prediction
error and DTDI (demand/baseline) behave under severe weather versus normal conditions —
the evidence behind Table \ref{tab:reliability} and Section 5.3's severe-weather
finding, self-contained here rather than only in notebook `07`.

## 1. Setup

In [1]:
import pandas as pd

from src.config import MODELS_DIR, TAXI_ZONE_LOOKUP_CSV
from src.evaluate import build_evaluation_frame
from src.features import split_train_test
from src.geospatial import load_processed_zone_hour
from src.modeling.train import load_model

processed_df = load_processed_zone_hour()
train_df, test_df = split_train_test(processed_df)

models = {
    name: load_model(MODELS_DIR / f"{name}.joblib")
    for name in ["linear_regression", "ridge", "gradient_boosting"]
}
eval_df = build_evaluation_frame(test_df, models)

zone_lookup = pd.read_csv(TAXI_ZONE_LOOKUP_CSV)[["LocationID", "Zone", "Borough"]].rename(
    columns={"LocationID": "pu_location_id", "Zone": "zone_name", "Borough": "borough"}
)
eval_df = eval_df.merge(zone_lookup, on="pu_location_id", how="left")
eval_df["pickup_date"] = eval_df["pickup_hour"].dt.date

print(f"Evaluation rows: {len(eval_df):,}")
print(f"Test period: {eval_df['pickup_hour'].min()} .. {eval_df['pickup_hour'].max()}")

2026-08-31 03:12:48.215 | INFO     | src.config:<module>:26 - PROJ_ROOT path is: /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final


Evaluation rows: 1,888,340
Test period: 2026-01-01 00:00:00 .. 2026-05-30 23:00:00


## 2. Recommendation 1 — rank zone-hours, don't just predict a number

Notebook `07`'s Table 4 already established Linear Regression (tied with Ridge) has the
best top-k overlap among all four models at every k except k=50, where Gradient Boosting
narrowly edges ahead. Here we isolate Linear Regression's own top-50 and top-1000
recovery rates against an unranked/random baseline of the same size.

In [2]:
from src.evaluate import top_k_zone_hour_overlap

n_total = len(eval_df)
for k in [50, 1000]:
    hit_rate = 100 * top_k_zone_hour_overlap(eval_df, k, "actual", "pred_linear_regression")
    random_rate = 100 * k / n_total
    print(
        f"k={k}: Linear Regression recovers {hit_rate:.0f}% of the true top-{k} "
        f"busiest zone-hours vs. a random baseline of {random_rate:.3f}% "
        f"({hit_rate / random_rate:.0f}x improvement)"
    )

k=50: Linear Regression recovers 38% of the true top-50 busiest zone-hours vs. a random baseline of 0.003% (14351x improvement)


k=1000: Linear Regression recovers 60% of the true top-1000 busiest zone-hours vs. a random baseline of 0.053% (1133x improvement)


**Single-day case study.** Identifying the busiest single day in the test period,
then comparing Linear Regression's predicted top-20 busiest zone-hours against the
actual top-20 for that day.

In [3]:
daily_totals = eval_df.loc[eval_df["taxi_type"] == "yellow"].groupby("pickup_date")["actual"].sum()
busiest_date = daily_totals.idxmax()
print(f"Busiest Yellow-taxi day in the test period: {busiest_date} ({daily_totals.max():,.0f} total pickups)")

Busiest Yellow-taxi day in the test period: 2026-04-25 (156,759 total pickups)


In [4]:
N = 20
day_df = eval_df[(eval_df["pickup_date"] == busiest_date) & (eval_df["taxi_type"] == "yellow")]

actual_top = day_df.nlargest(N, "actual")[["zone_name", "borough", "hour_of_day", "actual"]]
pred_top = day_df.nlargest(N, "pred_linear_regression")[
    ["zone_name", "borough", "hour_of_day", "pred_linear_regression"]
]
overlap = set(actual_top.index) & set(pred_top.index)

print(f"Top-{N} actual busiest zone-hours ({busiest_date}, Yellow):")
print(actual_top.reset_index(drop=True).to_string())
print(f"\nTop-{N} Linear Regression-predicted busiest zone-hours ({busiest_date}, Yellow):")
print(pred_top.reset_index(drop=True).to_string())
print(f"\nOverlap: {len(overlap)}/{N} zone-hours in common ({100 * len(overlap) / N:.0f}%)")

Top-20 actual busiest zone-hours (2026-04-25, Yellow):
                    zone_name    borough  hour_of_day  actual
0                East Village  Manhattan            1   826.0
1                East Village  Manhattan           23   767.0
2                East Village  Manhattan            0   739.0
3                East Village  Manhattan           22   627.0
4       Upper East Side South  Manhattan           15   606.0
5       Upper East Side South  Manhattan           16   590.0
6                West Village  Manhattan            1   576.0
7                East Village  Manhattan            2   569.0
8       Upper East Side North  Manhattan           14   564.0
9                West Village  Manhattan            0   548.0
10      Upper East Side North  Manhattan           15   544.0
11      Upper East Side South  Manhattan           17   540.0
12      Upper East Side South  Manhattan           14   527.0
13               West Village  Manhattan           23   516.0
14      Upper E

## 3. Recommendation 2 — treat severe weather as a reliability caution, not a demand signal

Severe weather does not spike demand — if anything the opposite — and none of the four
models reliably capture the shift that does happen. This section computes, across the
full test period: (a) each model's prediction error under severe vs. normal weather, (b)
actual DTDI (demand/baseline) under severe vs. normal weather, confirming demand is
suppressed not elevated, and (c) how much of that real shift each model's own
predictions reproduce, tied to how compressed each model's predicted DTDI already is
relative to actual demand variance. LASSO is refit here at its already-tuned alpha
(0.001, from notebook `07`) since it is not persisted to disk.

In [5]:
import numpy as np

from src.features import apply_standard_scaler, build_feature_matrix, fit_standard_scaler
from src.modeling.train import LASSO_PARAMS, fit_lasso_regression

X_train, y_train = build_feature_matrix(train_df, "linear")
X_test, y_test = build_feature_matrix(test_df, "linear")
scaler = fit_standard_scaler(X_train)
X_train_scaled = apply_standard_scaler(X_train, scaler)
X_test_scaled = apply_standard_scaler(X_test, scaler)

lasso_model = fit_lasso_regression(X_train_scaled, y_train, alpha=0.001, **LASSO_PARAMS)
lasso_pred = np.clip(np.expm1(lasso_model.predict(X_test_scaled)), a_min=0, a_max=None)
# Assigned positionally, not by label: eval_df's index was reset by the merge() in Setup,
# so aligning by X_test_scaled's original index would silently produce all-NaN.
eval_df["pred_lasso"] = lasso_pred
print("LASSO refit at alpha=0.001 (matches notebook 07's tuned value); predictions added.")
print(f"NaN count in pred_lasso: {eval_df['pred_lasso'].isna().sum()}")

LASSO refit at alpha=0.001 (matches notebook 07's tuned value); predictions added.
NaN count in pred_lasso: 0


In [6]:
from src.evaluate import regression_metrics_by_group

pred_cols = {
    "gradient_boosting": "pred_gradient_boosting",
    "linear_regression": "pred_linear_regression",
    "ridge": "pred_ridge",
    "lasso": "pred_lasso",
}

print("MAE/RMSE/R2 by weather severity, per model")
mae_ratios = {}
for name, col in pred_cols.items():
    metrics = regression_metrics_by_group(eval_df, "weather_severity", "actual", col)
    mae_ratios[name] = metrics.loc["severe", "MAE"] / metrics.loc["normal", "MAE"]
    print(f"\n{name}:")
    print(metrics.round(3))

print("\nMAE ratio (severe/normal), per model:")
for name, ratio in mae_ratios.items():
    print(f"  {name}: {ratio:.2f}x")

MAE/RMSE/R2 by weather severity, per model

gradient_boosting:
                    MAE    RMSE     R2        n
weather_severity                               
mild              2.306  10.627  0.915   322438
normal            2.120   9.578  0.942  1493840
severe            3.393  16.105  0.802    72062



linear_regression:
                    MAE    RMSE     R2        n
weather_severity                               
mild              2.395  10.829  0.912   322438
normal            2.216   9.615  0.942  1493840
severe            3.567  16.311  0.797    72062

ridge:
                    MAE    RMSE     R2        n
weather_severity                               
mild              2.395  10.829  0.912   322438
normal            2.216   9.615  0.942  1493840
severe            3.567  16.311  0.797    72062



lasso:
                    MAE    RMSE     R2        n
weather_severity                               
mild              2.429  11.018  0.909   322438
normal            2.263   9.864  0.939  1493840
severe            3.606  16.452  0.794    72062

MAE ratio (severe/normal), per model:
  gradient_boosting: 1.60x
  linear_regression: 1.61x
  ridge: 1.61x
  lasso: 1.59x


In [7]:
from src.evaluate import MIN_BASELINE_FOR_DTDI, compute_dtdi

reliable = eval_df[eval_df["baseline_pickup_count"] >= MIN_BASELINE_FOR_DTDI].copy()
reliable["dtdi_actual"] = compute_dtdi(reliable["actual"], reliable["baseline_pickup_count"])

dtdi_by_severity = reliable.groupby("weather_severity", observed=True)["dtdi_actual"].agg(
    ["mean", "median", "count"]
)
print("Actual DTDI by weather severity:")
print(dtdi_by_severity)

normal_dtdi = dtdi_by_severity.loc["normal", "mean"]
severe_dtdi = dtdi_by_severity.loc["severe", "mean"]
true_drop = normal_dtdi - severe_dtdi
print(f"\nMean actual DTDI: normal={normal_dtdi:.3f}, severe={severe_dtdi:.3f} (drop of {true_drop:.3f})")

actual_std = reliable["dtdi_actual"].std()
print(f"Actual DTDI std: {actual_std:.4f}\n")

print("Predicted DTDI by weather severity, per model, and how much of the true drop each recovers:")
for name, col in pred_cols.items():
    reliable[f"dtdi_pred_{name}"] = compute_dtdi(reliable[col], reliable["baseline_pickup_count"])
    pred_by_severity = reliable.groupby("weather_severity", observed=True)[f"dtdi_pred_{name}"].mean()
    pred_drop = pred_by_severity["normal"] - pred_by_severity["severe"]
    captured_pct = 100 * pred_drop / true_drop
    pred_std = reliable[f"dtdi_pred_{name}"].std()
    std_ratio = actual_std / pred_std
    print(
        f"  {name}: predicted DTDI normal={pred_by_severity['normal']:.3f}, "
        f"severe={pred_by_severity['severe']:.3f} (drop of {pred_drop:.3f}, "
        f"~{captured_pct:.0f}% of true drop) | std ratio (actual/predicted) = {std_ratio:.2f}x"
    )

Actual DTDI by weather severity:
                      mean    median   count
weather_severity                            
mild              0.993937  0.891720   98217
normal            1.000368  0.944099  456428
severe            0.897516  0.846922   22030

Mean actual DTDI: normal=1.000, severe=0.898 (drop of 0.103)
Actual DTDI std: 0.7619

Predicted DTDI by weather severity, per model, and how much of the true drop each recovers:
  gradient_boosting: predicted DTDI normal=0.920, severe=0.906 (drop of 0.015, ~14% of true drop) | std ratio (actual/predicted) = 4.89x
  linear_regression: predicted DTDI normal=0.939, severe=0.941 (drop of -0.002, ~-2% of true drop) | std ratio (actual/predicted) = 20.41x
  ridge: predicted DTDI normal=0.939, severe=0.941 (drop of -0.002, ~-2% of true drop) | std ratio (actual/predicted) = 20.41x
  lasso: predicted DTDI normal=0.934, severe=0.939 (drop of -0.005, ~-5% of true drop) | std ratio (actual/predicted) = 28.25x


## Notes

- Recommendation 1's top-50/top-1000 overlap figures are notebook `07`'s Table 4, cited
  not recomputed from scratch (only re-evaluated here for Linear Regression alone, which
  is all this notebook needs).
- The single-day case study is the one genuinely new computation for Recommendation 1.
- Recommendation 2's evidence (MAE by weather severity, actual/predicted DTDI by weather
  severity, and each model's DTDI compression ratio) is computed fresh here, independent
  of notebook `07` — this notebook loads its own models and refits LASSO itself, so it
  stays self-contained even though the underlying finding is the same one behind Table
  \ref{tab:reliability} in Section 5.3.
- The previous Recommendation 2 (weekly shift reallocation, Monday-vs-Thursday and
  weekend-vs-weekday midnight) was replaced: severe weather's effect on prediction
  reliability is a stronger, more model-grounded finding, and the day-of-week/hour
  pattern it replaced is still fully documented in notebook `05`'s Section 3.2 EDA, just
  no longer used as a Recommendation.